In [ ]:
# ==============================================================================
# [GLOBAL CELL 1] KONFIGURASI SENTRAL & SEEDING (RECOVERY FINE-TUNING)
# ==============================================================================
import os
import random
from enum import Enum
from typing import Tuple, Dict

import numpy as np
import tensorflow as tf

# ------------------------------------------------------------------------------
# 1. ENUMERASI ARSITEKTUR & METRIK
# ------------------------------------------------------------------------------
class ModelArch(str, Enum):
    MOBILENET_V3 = 'MobileNetV3Small'
    EFFICIENTNET_B0 = 'EfficientNetB0'
    RESNET_50 = 'ResNet50'

class PruningMetric(str, Enum):
    L1_NORM = "l1_norm"
    L2_NORM = "l2_norm"

# ------------------------------------------------------------------------------
# 2. SENTRALISASI KONFIGURASI (Standar 2 & 4)
# ------------------------------------------------------------------------------
class RecoveryConfig:
    """Konfigurasi utama untuk Fase Recovery Fine-Tuning Pasca-Pruning."""
    
    # Dataset & Dimensi (Membutuhkan Train & Val Set penuh)
    PATH_TRAIN_FFB: str = "/kaggle/input/datasets/marrioqqqq/dataset-sawit/dataset-sawit/train"
    PATH_VAL_FFB: str = "/kaggle/input/datasets/marrioqqqq/dataset-sawit/dataset-sawit/val"
    IMG_SIZE: Tuple[int, int] = (224, 224)
    
    # --------------------------------------------------------------------------
    # TARGET PARETO-OPTIMAL (Tentukan rasio pemotongan final di sini)
    # --------------------------------------------------------------------------
    PARETO_RATIOS: Dict[ModelArch, float] = {
        ModelArch.MOBILENET_V3: 0.40,   # Sisa 60% parameter aktif
        ModelArch.EFFICIENTNET_B0: 0.50 # Sesuaikan dengan temuan grafik Anda
    }
    PRUNING_METRIC: PruningMetric = PruningMetric.L2_NORM
    EPSILON_NORM: float = 1e-9
    
    # # Hyperparameter Training Pemulihan (SGDR)
    # RECOVERY_EPOCHS: int = 100
    # PATIENCE_EARLY_STOPPING: int = 15

    # Configurasi Debug
    RECOVERY_EPOCHS: int = 2
    PATIENCE_EARLY_STOPPING: int = 15

    SGDR_LR_MULTIPLIER: float = 0.1 # Base LR turun 90% dari nilai HPO untuk mencegah Gradient Shock
    
    # --------------------------------------------------------------------------
    # PATH INTEGRASI LINTAS NOTEBOOK
    # --------------------------------------------------------------------------
    # 1. Path ke HPO Output (Untuk menyedot Optimizer, Weight Decay, & Batch Size)
    PATH_HPO_OUTPUTS: str = "/kaggle/input/datasets/marrioqqqq/hpo-output"
    
    # 2. Path ke Model FP32 Baseline (Untuk dipotong ulang secara definitif)
    PATH_TRAINED_MODELS: str = "/kaggle/input/models/marrioqqqq/prune-model-ffb/keras/default/1"
    
    # 3. Path Output Dinamis Notebook Ini
    BASE_OUTPUT_DIR: str = "/kaggle/working/recovery_outputs"

    @classmethod
    def get_output_dir(cls, arch: ModelArch) -> str:
        target_dir = os.path.join(cls.BASE_OUTPUT_DIR, arch.value)
        os.makedirs(target_dir, exist_ok=True)
        return target_dir

    @classmethod
    def get_hpo_params_path(cls, arch: ModelArch) -> str:
        """Mengambil JSON parameter HPO."""
        return os.path.join(cls.PATH_HPO_OUTPUTS, arch.value, f"best_params_{arch.value}.json")

    @classmethod
    def get_input_model_path(cls, arch: ModelArch) -> str:
        """Mengambil model FP32 Baseline menggunakan sistem Auto-Detect."""
        path_flat = os.path.join(cls.PATH_TRAINED_MODELS, f"model_{arch.value}_phase2.keras")
        path_nested = os.path.join(cls.PATH_TRAINED_MODELS, arch.value, f"model_{arch.value}_phase2.keras")
        return path_flat if os.path.exists(path_flat) else path_nested

    @classmethod
    def get_db_path(cls, arch: ModelArch) -> str:
        """Database khusus untuk mencatat dinamika metrik dan SGDR Learning Rate."""
        return os.path.join(cls.get_output_dir(arch), f"recovery_log_{arch.value}.db")
        
    @classmethod
    def get_final_model_path(cls, arch: ModelArch) -> str:
        """Nama file model akhir yang siap dikuantisasi."""
        ratio_pct = int(cls.PARETO_RATIOS.get(arch, 0) * 100)
        return os.path.join(cls.get_output_dir(arch), f"model_{arch.value}_pruned_{ratio_pct}pct_recovered.keras")

    @classmethod
    def get_unrecovered_model_path(cls, arch: ModelArch) -> str:
        """Menyimpan model yang baru saja dipotong SEBELUM di-recovery (Zero-Shot)."""
        ratio_pct = int(cls.PARETO_RATIOS.get(arch, 0) * 100)
        return os.path.join(cls.get_output_dir(arch), f"model_{arch.value}_pruned_{ratio_pct}pct_unrecovered.keras")

    @classmethod
    def get_tflite_fp32_path(cls, arch: ModelArch) -> str:
        """Standar 4: Path dinamis untuk ekspor TFLite FP32 dari model Recovered."""
        ratio_pct = int(cls.PARETO_RATIOS.get(arch, 0) * 100)
        return os.path.join(cls.get_output_dir(arch), f"model_{arch.value}_pruned_{ratio_pct}pct_recovered_fp32.tflite")
        
# ------------------------------------------------------------------------------
# 3. REPRODUKSIBILITAS (Standar 3)
# ------------------------------------------------------------------------------
def set_deterministic_seed(seed: int = 42) -> None:
    os.environ['PYTHONHASHSEED'] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    tf.keras.utils.set_random_seed(seed)
    tf.config.experimental.enable_op_determinism()
    print(f"[SETUP] Deterministic Seed dikunci pada: {seed}")

set_deterministic_seed(42)
print(f"[SETUP] TensorFlow Version: {tf.__version__}")

In [ ]:
# ==============================================================================
# [CELL 2] KERAS SERIALIZATION PATCH & RECOVERY SQLITE LOGGER
# ==============================================================================
import sqlite3
from typing import Dict, Any
from tensorflow.keras import layers, callbacks

# ------------------------------------------------------------------------------
# 1. KOREKSI SERIALISASI KERAS (MONKEY PATCHING)
# ------------------------------------------------------------------------------
def patch_keras_serialization() -> None:
    if hasattr(layers.Conv2D, '_is_patched_for_pruning'):
        return
        
    _orig_dense = layers.Dense.__init__
    _orig_conv2d = layers.Conv2D.__init__
    _orig_dwconv2d = layers.DepthwiseConv2D.__init__
    _orig_bn = layers.BatchNormalization.__init__

    def safe_dense(self, *args, **kwargs): kwargs.pop('quantization_config', None); _orig_dense(self, *args, **kwargs)
    def safe_conv2d(self, *args, **kwargs): kwargs.pop('quantization_config', None); _orig_conv2d(self, *args, **kwargs)
    def safe_dwconv2d(self, *args, **kwargs): kwargs.pop('quantization_config', None); _orig_dwconv2d(self, *args, **kwargs)
    def safe_bn(self, *args, **kwargs): kwargs.pop('quantization_config', None); _orig_bn(self, *args, **kwargs)

    layers.Dense.__init__ = safe_dense
    layers.Conv2D.__init__ = safe_conv2d
    layers.DepthwiseConv2D.__init__ = safe_dwconv2d
    layers.BatchNormalization.__init__ = safe_bn
    layers.Conv2D._is_patched_for_pruning = True
    print("[SETUP] Keras Layer Serialization berhasil di-patch untuk Re-Pruning.")

patch_keras_serialization()

# ------------------------------------------------------------------------------
# 2. SQLITE LOGGER BASE CLASS (Standar 8 & 9)
# ------------------------------------------------------------------------------
class RecoverySQLiteLogger(callbacks.Callback):
    """
    Custom Keras Callback.
    Merekam metrik per epoch (termasuk jejak SGDR Learning Rate) langsung ke SQLite.
    Menggantikan peran CSVLogger lama.
    """
    def __init__(self, db_path: str):
        super().__init__()
        self.db_path = db_path
        self._initialize_db()

    def _initialize_db(self):
        conn = sqlite3.connect(self.db_path)
        cursor = conn.cursor()
        cursor.execute('''
            CREATE TABLE IF NOT EXISTS recovery_logs (
                epoch INTEGER PRIMARY KEY,
                loss REAL, accuracy REAL,
                val_loss REAL, val_accuracy REAL,
                learning_rate REAL
            )
        ''')
        conn.commit()
        conn.close()

    def on_epoch_end(self, epoch: int, logs: Dict[str, Any] = None):
        logs = logs or {}
        
        # Ekstrak learning rate dinamis dari CosineDecayRestarts (SGDR)
        lr = self.model.optimizer.learning_rate
        if isinstance(lr, tf.keras.optimizers.schedules.LearningRateSchedule):
            lr_value = float(lr(self.model.optimizer.iterations).numpy())
        else:
            lr_value = float(lr.numpy() if hasattr(lr, 'numpy') else lr)

        conn = sqlite3.connect(self.db_path)
        cursor = conn.cursor()
        cursor.execute('''
            INSERT OR REPLACE INTO recovery_logs 
            (epoch, loss, accuracy, val_loss, val_accuracy, learning_rate)
            VALUES (?, ?, ?, ?, ?, ?)
        ''', (
            epoch + 1,
            logs.get('loss', 0.0), logs.get('accuracy', 0.0),
            logs.get('val_loss', 0.0), logs.get('val_accuracy', 0.0),
            lr_value
        ))
        conn.commit()
        conn.close()

print("[BERHASIL] Keras Patch & Recovery SQLite Logger siap.")

In [ ]:
# ==============================================================================
# [CELL 3] DEFINITIVE PRUNER: PARETO-OPTIMAL SLICING UTILITY
# ==============================================================================
import re
import numpy as np
import tensorflow as tf
from typing import Dict, Any

class ParetoPruner:
    """
    Standar 5 & 9: Utilitas Agnostik untuk memotong arsitektur secara definitif.
    Menggabungkan fungsi Planner dan Surgeon khusus untuk eksekusi satu tembakan (One-Shot).
    """
    def __init__(self, config: RecoveryConfig, architecture: ModelArch):
        self.cfg = config
        self.arch = architecture

    def _calculate_energy(self, weights: np.ndarray, metric: PruningMetric, axis: tuple) -> np.ndarray:
        if metric == PruningMetric.L2_NORM: return np.sum(np.square(weights), axis=axis)
        elif metric == PruningMetric.L1_NORM: return np.sum(np.abs(weights), axis=axis)
        raise ValueError(f"Metrik {metric} belum didukung.")

    def _plan_normalized(self, model: tf.keras.Model, target_ratio: float) -> Dict[str, Dict[str, Any]]:
        blocks: Dict[str, Dict[str, tf.keras.layers.Layer]] = {}
        regex_pattern = r'(_expand_conv|_dwconv|_se_reduce|_se_expand|_project_conv)' if self.arch == ModelArch.EFFICIENTNET_B0 else r'(_expand|_depthwise|_squeeze|_project)'
        
        for layer in model.layers:
            match = re.split(regex_pattern, layer.name)
            if len(match) > 1:
                prefix = match[0]
                if prefix not in blocks: blocks[prefix] = {}
                
                if ("_expand" in layer.name or "_expand_conv" in layer.name) and "bn" not in layer.name: blocks[prefix]['expand'] = layer
                elif ("_depthwise" in layer.name or "_dwconv" in layer.name) and "bn" not in layer.name: blocks[prefix]['depthwise'] = layer
                elif ("project" in layer.name or "_project_conv" in layer.name) and "bn" not in layer.name: blocks[prefix]['project'] = layer

        plans: Dict[str, Dict[str, Any]] = {}
        for prefix, components in blocks.items():
            if 'expand' not in components: continue
            
            w_expand = components['expand'].get_weights()
            if not w_expand: continue
            num_filters: int = w_expand[0].shape[-1]
            
            raw_energy = self._calculate_energy(w_expand[0], self.cfg.PRUNING_METRIC, axis=(0, 1, 2))
            norm_energy = raw_energy / (np.max(raw_energy) + self.cfg.EPSILON_NORM)
            
            if 'depthwise' in components and components['depthwise'].get_weights():
                dw_energy = self._calculate_energy(components['depthwise'].get_weights()[0], self.cfg.PRUNING_METRIC, axis=(0, 1, 3))
                norm_energy += (dw_energy / (np.max(dw_energy) + self.cfg.EPSILON_NORM))
                    
            if 'project' in components and components['project'].get_weights():
                proj_energy = self._calculate_energy(components['project'].get_weights()[0], self.cfg.PRUNING_METRIC, axis=(0, 1, 3))
                norm_energy += (proj_energy / (np.max(proj_energy) + self.cfg.EPSILON_NORM))

            n_prune = int(num_filters * target_ratio)
            if n_prune == 0: continue
            n_keep = max(1, num_filters - n_prune)
            keep_indices = np.sort(np.argsort(norm_energy)[-n_keep:])
            
            plans[prefix] = {
                'mask': keep_indices, 
                'surviving_filters': len(keep_indices),
                'ratio': (n_prune / num_filters)
            }
        return plans

    def slice_model(self, model: tf.keras.Model, target_ratio: float) -> tf.keras.Model:
        """Fungsi utama untuk menghasilkan model terpotong secara fisik."""
        plans = self._plan_normalized(model, target_ratio)
        config = model.get_config()
        
        for l_conf in config['layers']:
            l_name = l_conf['name']
            l_conf.pop('build_config', None)
            if 'config' in l_conf:
                l_conf['config'].pop('batch_input_shape', None)
                l_conf['config'].pop('input_shape', None)

            matched_prefix = next((p for p in sorted(plans.keys(), key=len, reverse=True) if l_name.startswith(p)), None)
            if matched_prefix:
                p = plans[matched_prefix]
                if l_name.endswith(("_expand", "_expand_conv", "se_expand")): l_conf['config']['filters'] = p['surviving_filters']
                elif "squeeze_excite_conv_1" in l_name: l_conf['config']['filters'] = p['surviving_filters']
                elif "squeeze_excite_conv" in l_name and "conv_1" not in l_name: 
                    l_conf['config']['filters'] = max(1, int(l_conf['config']['filters'] * (1 - p['ratio'])))
                elif "_se_reshape" in l_name: l_conf['config']['target_shape'] = (1, 1, p['surviving_filters'])

        pruned_model = tf.keras.Model.from_config(config)

        for new_layer in pruned_model.layers:
            try: old_layer = model.get_layer(new_layer.name)
            except ValueError: continue
            if not old_layer.weights: continue
            old_w = old_layer.get_weights()
            matched_prefix = next((p for p in sorted(plans.keys(), key=len, reverse=True) if new_layer.name.startswith(p)), None)
            
            if not matched_prefix or new_layer.weights[0].shape == old_w[0].shape: 
                new_layer.set_weights(old_w)
                continue
                
            mask, name = plans[matched_prefix]['mask'], new_layer.name
            
            if ("_expand" in name or "_se_expand" in name) and "project" not in name and isinstance(new_layer, tf.keras.layers.Conv2D):
                w, b = old_w[0][:, :, :, mask], old_w[1][mask] if len(old_w) > 1 else None
                new_layer.set_weights([w, b] if b is not None else [w])
            elif "squeeze_excite_conv_1" in name:
                in_ch = new_layer.weights[0].shape[2]
                w, b = old_w[0][:, :, :in_ch, mask], old_w[1][mask] if len(old_w) > 1 else None
                if w.shape == new_layer.weights[0].shape: new_layer.set_weights([w, b] if b is not None else [w])
            elif "squeeze_excite_conv" in name:
                target_out = new_layer.filters
                w, b = old_w[0][:, :, mask, :target_out], old_w[1][:target_out] if len(old_w) > 1 else None
                if w.shape == new_layer.weights[0].shape: new_layer.set_weights([w, b] if b is not None else [w])
            elif "_se_reduce" in name:
                w, b = old_w[0][:, :, mask, :], old_w[1] if len(old_w) > 1 else None
                new_layer.set_weights([w, b] if b is not None else [w])
            elif isinstance(new_layer, tf.keras.layers.DepthwiseConv2D):
                w, b = old_w[0][:, :, mask, :], old_w[1][mask] if len(old_w) > 1 else None
                new_layer.set_weights([w, b] if b is not None else [w])
            elif ("project" in name and isinstance(new_layer, tf.keras.layers.Conv2D)) or (isinstance(new_layer, tf.keras.layers.BatchNormalization) and "project" not in name):
                if "project_bn" in name: new_layer.set_weights(old_w)
                elif isinstance(new_layer, tf.keras.layers.BatchNormalization): new_layer.set_weights([x[mask] for x in old_w])
                else:
                    w, b = old_w[0][:, :, mask, :], old_w[1] if len(old_w) > 1 else None
                    new_layer.set_weights([w, b] if b is not None else [w])
            else: new_layer.set_weights(old_w)
            
        return pruned_model

print("[BERHASIL] Cell 3 (ParetoPruner Utility) dimuat.")

In [ ]:
# ==============================================================================
# [CELL 4] RECOVERY ENGINE: SGDR, BN-PROTECTION, & FINE-TUNING ORCHESTRATOR
# ==============================================================================
import os
import json
import tensorflow as tf
from tensorflow.keras import models, optimizers, callbacks

class RecoveryEngine:
    """
    Standar 9: Mesin pemulihan agnostik.
    Menyerap JSON HPO, memotong model, mengunci BatchNormalization, 
    serta mengeksekusi pelatihan SGDR dengan aman.
    """
    def __init__(self, config: RecoveryConfig, architecture: ModelArch):
        self.cfg = config
        self.arch = architecture
        self.pareto_ratio = self.cfg.PARETO_RATIOS.get(self.arch, 0.0)
        
        self.db_path = self.cfg.get_db_path(self.arch)
        self.final_model_path = self.cfg.get_final_model_path(self.arch)
        
        self._load_hpo_params()
        self._build_datasets()
        
    def _load_hpo_params(self) -> None:
        """Standar 2: Menyerap parameter pelatihan dari JSON tahap HPO."""
        hpo_path = self.cfg.get_hpo_params_path(self.arch)
        if not os.path.exists(hpo_path):
            raise FileNotFoundError(f"[ERROR] Hasil HPO JSON tidak ditemukan di: {hpo_path}")
            
        with open(hpo_path, 'r') as f:
            self.hpo_params = json.load(f)
        print(f"[INFO] HPO Params termuat untuk {self.arch.value}: BS={self.hpo_params['batch_size']}, Opt={self.hpo_params['optimizer']}")

    def _build_datasets(self) -> None:
        batch_size = self.hpo_params['batch_size']
        
        self.ds_train = tf.keras.utils.image_dataset_from_directory(
            self.cfg.PATH_TRAIN_FFB, image_size=self.cfg.IMG_SIZE, 
            batch_size=batch_size, shuffle=True, label_mode='int', verbose=0
        ).take(1).prefetch(tf.data.AUTOTUNE)
        
        self.ds_val = tf.keras.utils.image_dataset_from_directory(
            self.cfg.PATH_VAL_FFB, image_size=self.cfg.IMG_SIZE, 
            batch_size=batch_size, shuffle=False, label_mode='int', verbose=0
        ).take(1).prefetch(tf.data.AUTOTUNE)
        
        self.steps_per_epoch = len(self.ds_train)

    def _export_tflite_fp32(self, keras_path: str) -> None:
        """Fungsi internal untuk mengekspor model Recovered ke TFLite FP32 secara otomatis."""
        tflite_path = self.cfg.get_tflite_fp32_path(self.arch)
        print(f"\n[INFO] Mengonversi arsitektur Recovered ke format TFLite FP32 murni...")
        
        model = models.load_model(keras_path)
        converter = tf.lite.TFLiteConverter.from_keras_model(model)
        
        tflite_model = converter.convert()
        with open(tflite_path, 'wb') as f:
            f.write(tflite_model)
            
        size_mb = os.path.getsize(tflite_path) / (1024 * 1024)
        print(f"[BERHASIL] TFLite FP32 Recovered Tersimpan: {tflite_path} (Ukuran: {size_mb:.2f} MB)")
        
    def execute_recovery(self) -> None:
        print("\n" + "═"*75)
        print(f"🩺 MEMULAI RECOVERY: {self.arch.value} | TARGET PRUNING: {int(self.pareto_ratio*100)}%")
        print("═"*75)
        
        # 1. Muat & Potong Model
        fp32_path = self.cfg.get_input_model_path(self.arch)
        if not os.path.exists(fp32_path):
            raise FileNotFoundError(f"[ERROR] Model FP32 tidak ditemukan: {fp32_path}")
            
        print("[INFO] Memuat dan memotong arsitektur FP32...")
        baseline_model = models.load_model(fp32_path)
        
        pruner = ParetoPruner(self.cfg, self.arch)
        recovered_model = pruner.slice_model(baseline_model, self.pareto_ratio)

        unrecovered_path = self.cfg.get_unrecovered_model_path(self.arch)
        recovered_model.save(unrecovered_path)
        print(f"[INFO] 📦 Model Unrecovered (Zero-Shot) disimpan ke: {unrecovered_path}")
        
        # 2. Buka Gembok & Proteksi BatchNormalization
        frozen_bn_count = 0
        for layer in recovered_model.layers:
            layer.trainable = True
            if isinstance(layer, tf.keras.layers.BatchNormalization):
                layer.trainable = False
                frozen_bn_count += 1
                
        print(f"[INFO] Proteksi BN Aktif: {frozen_bn_count} layer BatchNormalization dibekukan.")
        print(f"[INFO] Sisa Parameter Aktif: {recovered_model.count_params():,}")
        
        # 3. Setup SGDR (Cosine Annealing) dengan Base LR diturunkan 90% (Multiplier 0.1)
        base_lr = self.hpo_params['learning_rate'] * self.cfg.SGDR_LR_MULTIPLIER
        weight_decay = self.hpo_params['weight_decay']
        optimizer_type = self.hpo_params['optimizer']
        
        lr_scheduler = optimizers.schedules.CosineDecayRestarts(
            initial_learning_rate=base_lr,
            first_decay_steps=self.steps_per_epoch * 5, 
            t_mul=1.5, m_mul=0.8, alpha=1e-6
        )
        
        if optimizer_type == 'adamw':
            opt = optimizers.AdamW(learning_rate=lr_scheduler, weight_decay=weight_decay)
        elif optimizer_type == 'adam':
            opt = optimizers.Adam(learning_rate=lr_scheduler, weight_decay=weight_decay)
        else:
            opt = optimizers.SGD(learning_rate=lr_scheduler, momentum=0.9, weight_decay=weight_decay)
            
        recovered_model.compile(optimizer=opt, loss='sparse_categorical_crossentropy', metrics=['accuracy'])
        
        # 4. Callbacks & Training Eksekusi
        callbacks_list = [
            callbacks.ModelCheckpoint(filepath=self.final_model_path, save_best_only=True, monitor='val_loss', mode='min', verbose=1),
            callbacks.EarlyStopping(monitor='val_loss', patience=self.cfg.PATIENCE_EARLY_STOPPING, restore_best_weights=True, verbose=1),
            RecoverySQLiteLogger(self.db_path)
        ]
        
        print(f"\n[INFO] Memulai Fisioterapi (Max {self.cfg.RECOVERY_EPOCHS} Epochs)...")
        recovered_model.fit(
            self.ds_train, validation_data=self.ds_val,
            epochs=self.cfg.RECOVERY_EPOCHS, callbacks=callbacks_list
        )
        
        # 5. Evaluasi Akhir
        final_loss, final_acc = recovered_model.evaluate(self.ds_val, verbose=0)
        print(f"\n[BERHASIL] Recovery Selesai! Model Final diekspor ke: {self.final_model_path}")
        print(f"[METRIK AKHIR] Akurasi: {final_acc*100:.2f}% | Loss: {final_loss:.4f}")

        self._export_tflite_fp32(self.final_model_path)

print("[BERHASIL] Cell 4 (Recovery Engine) dimuat.")

In [ ]:
# ==============================================================================
# [CELL 5] RECOVERY EXECUTOR: BATCH LOOP MODE
# ==============================================================================
import gc
import tensorflow as tf

config = RecoveryConfig()

# Seleksi Arsitektur (Beri # jika ingin melompati arsitektur tertentu)
MODELS_TO_RECOVER = [
    ModelArch.MOBILENET_V3,
    # ModelArch.EFFICIENTNET_B0,
    # ModelArch.RESNET_50
]

print("\n" + "★"*75)
print(f"🔥 MEMULAI RECOVERY FINE-TUNING UNTUK {len(MODELS_TO_RECOVER)} ARSITEKTUR")
print("★"*75)

for idx, arch in enumerate(MODELS_TO_RECOVER, start=1):
    print(f"\n{'='*75}")
    print(f"[{idx}/{len(MODELS_TO_RECOVER)}] EKSEKUSI RECOVERY: {arch.value}")
    print(f"{'='*75}")
    
    try:
        # Reset State & Seed untuk jaminan reproduksibilitas absolut
        tf.keras.backend.clear_session()
        gc.collect()
        set_deterministic_seed(42)
        
        # Inisialisasi & Eksekusi Engine
        engine = RecoveryEngine(config=config, architecture=arch)
        engine.execute_recovery()
        
        # Pembersihan Agresif Pasca-Training
        print(f"\n[CLEANUP] Membersihkan VRAM GPU pasca-Recovery untuk {arch.value}...")
        del engine
        gc.collect()
        
    except FileNotFoundError as e:
        print(f"\n[SKIP] Melompati {arch.value} karena dependensi hilang: {e}")
    except Exception as e:
        print(f"\n[CRITICAL ERROR] Eksekusi Recovery gagal pada {arch.value}: {e}")

print("\n" + "★"*75)
print("🏆 SELURUH PROSES RECOVERY FINE-TUNING SELESAI!")
print("★"*75)

In [ ]:
# ==============================================================================
# [CELL 6] RECOVERY VISUALIZER: DYNAMICS & SGDR TRACKING (SQLITE BACKEND)
# ==============================================================================
import os
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, FileLink

class RecoveryVisualizer:
    """
    Standar 9: Base Class Agnostik untuk merender grafik dari SQLite Recovery Log.
    Menghasilkan 3 subplot (Akurasi, Loss, dan Fluktuasi SGDR) dalam satu kanvas.
    """
    def __init__(self, config: RecoveryConfig, architecture: ModelArch):
        self.cfg = config
        self.arch = architecture
        self.db_path = self.cfg.get_db_path(self.arch)
        self.base_output_dir = self.cfg.get_output_dir(self.arch)
        
        # Standard 4: Isolasi Struktur Direktori
        self.vis_dir = os.path.join(self.base_output_dir, "visualisation")

        os.makedirs(self.vis_dir, exist_ok=True)
        
        # Path File Ekspor
        self.p_dynamics = os.path.join(self.vis_dir, f"recovery_{self.arch.value}_dynamics.png")
        
        print(f"\n" + "="*60)
        print(f"📊 INISIALISASI VISUALIZER RECOVERY: {self.arch.value}")
        print("="*60)
        
        self._validate_database()
        self.df = self._load_data()

    # --------------------------------------------------------------------------
    # DATA EXTRACTION (Standar 8)
    # --------------------------------------------------------------------------
    def _validate_database(self) -> None:
        if not os.path.exists(self.db_path):
            raise FileNotFoundError(f"[ERROR] Database {self.db_path} tidak ditemukan.")
        if os.path.getsize(self.db_path) == 0:
            raise ValueError(f"[ERROR] Database {self.db_path} kosong.")
            
    def _load_data(self) -> pd.DataFrame:
        conn = sqlite3.connect(self.db_path)
        try:
            df = pd.read_sql_query("SELECT * FROM recovery_logs ORDER BY epoch", conn)
        except sqlite3.OperationalError:
            df = pd.DataFrame()
        conn.close()
        
        if df.empty:
            raise ValueError(f"[ERROR] Tabel 'recovery_logs' kosong pada arsitektur {self.arch.value}.")
        return df

    # --------------------------------------------------------------------------
    # FUNGSI RENDER: 3-in-1 PLOT (ACC, LOSS, SGDR)
    # --------------------------------------------------------------------------
    def plot_recovery_dynamics(self) -> None:
        """Merender dinamika pemulihan model dalam 3 panel grafik sejajar."""
        plt.style.use('default')
        sns.set_theme(style="whitegrid", context="paper", font_scale=1.1)
        
        fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(18, 5))
        
        # Subplot 1: Dinamika Akurasi
        ax1.plot(self.df['epoch'], self.df['accuracy'], label='Train Accuracy', color='#3498db', linewidth=2)
        ax1.plot(self.df['epoch'], self.df['val_accuracy'], label='Val Accuracy', color='#e74c3c', linewidth=2.5, linestyle='--')
        ax1.set_title(f'Dinamika Akurasi Pemulihan ({self.arch.value})', fontweight='bold', pad=10)
        ax1.set_xlabel('Epoch', fontweight='bold')
        ax1.set_ylabel('Akurasi', fontweight='bold')
        ax1.legend()
        ax1.grid(True, linestyle='--', alpha=0.6)

        # Subplot 2: Dinamika Loss
        ax2.plot(self.df['epoch'], self.df['loss'], label='Train Loss', color='#2ecc71', linewidth=2)
        ax2.plot(self.df['epoch'], self.df['val_loss'], label='Val Loss', color='#f39c12', linewidth=2.5, linestyle='--')
        ax2.set_title(f'Dinamika Loss Pemulihan ({self.arch.value})', fontweight='bold', pad=10)
        ax2.set_xlabel('Epoch', fontweight='bold')
        ax2.set_ylabel('Loss', fontweight='bold')
        ax2.legend()
        ax2.grid(True, linestyle='--', alpha=0.6)

        # Subplot 3: Pergerakan SGDR (Learning Rate)
        ax3.plot(self.df['epoch'], self.df['learning_rate'], label='Learning Rate', color='#9b59b6', linewidth=2)
        ax3.set_title(f'Jejak SGDR Cosine Annealing ({self.arch.value})', fontweight='bold', pad=10)
        ax3.set_xlabel('Epoch', fontweight='bold')
        ax3.set_ylabel('Learning Rate (Log Scale)', fontweight='bold')
        ax3.set_yscale('log')
        ax3.legend()
        ax3.grid(True, which="both", linestyle='--', alpha=0.6)

        fig.tight_layout()
        plt.savefig(self.p_dynamics, dpi=300, bbox_inches='tight')
        plt.close()

    def run_all_visualisations(self) -> None:
        self.plot_recovery_dynamics()
        print(f"[BERHASIL] Grafik Recovery Resolusi Tinggi Tersimpan di: {self.vis_dir}")
        display(FileLink(self.p_dynamics))

print("[BERHASIL] Cell 6 (Class RecoveryVisualizer) dimuat.")

In [ ]:
# ==============================================================================
# [CELL 7] RECOVERY VISUALIZATION EXECUTOR (BATCH RUNNER)
# ==============================================================================

# Mewarisi variabel MODELS_TO_RECOVER dari Cell 5 jika dieksekusi berurutan.
# Gunakan fallback jika cell ini dijalankan terpisah (misalnya di hari lain).
try:
    models_to_visualize = MODELS_TO_RECOVER
except NameError:
    models_to_visualize = [ModelArch.MOBILENET_V3] 

print("\n" + "★"*75)
print(f"🎨 MEMULAI RENDER VISUALISASI RECOVERY UNTUK {len(models_to_visualize)} ARSITEKTUR")
print("★"*75)

config_vis = RecoveryConfig()

for arch in models_to_visualize:
    try:
        # Inisialisasi visualizer untuk arsitektur spesifik
        visualizer = RecoveryVisualizer(config=config_vis, architecture=arch)
        
        # Render dan simpan grafik 3-in-1
        visualizer.run_all_visualisations()
        
    except FileNotFoundError as e:
        print(e)
    except ValueError as e:
        print(e)
    except Exception as e:
        print(f"\n[CRITICAL ERROR] Gagal merender grafik untuk {arch.value}: {e}")

print("\n" + "★"*75)
print("🏆 SELURUH PROSES RENDER GRAFIK RECOVERY SELESAI!")
print("★"*75)

In [ ]:
# ==============================================================================
# [CELL 8] DEBUG: INSPEKSI STRUKTUR DIREKTORI OUTPUT RECOVERY
# ==============================================================================
import os

def print_directory_tree(startpath: str) -> None:
    print(f"\n🌳 STRUKTUR DIREKTORI RECOVERY: {startpath}")
    print("="*60)
    
    if not os.path.exists(startpath):
        print(f"[WARNING] Direktori {startpath} belum dibuat.")
        return

    for root, dirs, files in os.walk(startpath):
        level = root.replace(startpath, '').count(os.sep)
        indent = ' ' * 4 * level
        print(f"{indent}📂 {os.path.basename(root)}/")
        
        subindent = ' ' * 4 * (level + 1)
        for f in files:
            if f.endswith('.keras'): icon = "📦"
            elif f.endswith('.db'): icon = "🗄️"
            elif f.endswith('.png'): icon = "🖼️"
            else: icon = "📄"
            print(f"{subindent}{icon} {f}")

config_debug = RecoveryConfig()
print_directory_tree(config_debug.BASE_OUTPUT_DIR)

In [ ]:
import shutil
from IPython.display import FileLink, display

# Folder yang ingin didownload
folder_path = "/kaggle/working/recovery_outputs"

# Nama file ZIP output
zip_path = "/kaggle/working/recovery_outputs.zip"

# Buat ZIP
shutil.make_archive(
    base_name=zip_path.replace(".zip", ""),
    format="zip",
    root_dir=folder_path
)

print(f"ZIP berhasil dibuat: {zip_path}")

# Tampilkan link download
display(FileLink(zip_path))